In [ ]:
Task1_Misinformation_Prompt_merged= f""" 

You are a medical expert, specialized in identifying cancer-related misinformation. 
Definition:
Cancer misinformation refers to any information about cancer, its risk factors, prevention, diagnosis, or treatment that is false, inaccurate, misleading, or not supported by current scientific evidence or consensus, and that may lead to harmful consequences. This includes:
Unproven or ineffective treatments
Misrepresentation or selective use of facts
Outdated or decontextualized medical information
Conspiracy narratives (e.g., hidden cures)
Promotional or commercial content presented as medical advice

Task: Analyze the text provided below. Classify it as either:
'Yes' if the text contains misinformation
'No' if the text does not contain misinformation or if there is insufficient information


Output Format:
Label: 'Yes' or 'No' . Don't use additional text, and don't use [] around the label
Evidence: If classified as 1, quote the specific part of the text that suggests or implies misinformation and explain why it contradicts scientific evidence.
If classified as 1, explain your step-by-step reasoning to conclude that the text contains misinformation



""" 

In [ ]:
import time
from tqdm import tqdm
from openai import OpenAI

# 1. Define your list of models
GPT_Models = [
    "gpt-5.5-2026-04-23",
    "gpt-5.4-2026-03-05",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.2-2025-12-11",
    "gpt-5.1-2025-11-13",
    "gpt-5-2025-08-07",
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4.1-2025-04-14",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4.1-nano-2025-04-14",
    "gpt-4o-2024-11-20",
    "gpt-4o-mini-2024-07-18"
]

def classify_misinfo_new_prompt(text, model_name, use_temperature=True):
    retries = 2
    while retries > 0:
        try:
            kwargs = {
                "model": model_name,
                "messages": [{"role": "user", "content": Task1_Misinformation_Prompt_merged + text}]
            }
            if use_temperature:
                kwargs["temperature"] = 0.0
                
            completion = client.chat.completions.create(**kwargs)
            return completion.choices[0].message.content
            
        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)
            
    return "API_ERROR"

# 2. Loop through each model, run configuration check, and execute
tqdm.pandas()
# output_filename = "Camera_Ready_GT_set_experiments_results.csv"

file_path = r'/Users/ap3943/Camera_Ready_GT_set_experiments_results.csv'


# 1. Read the CSV file
df_q1 = pd.read_csv(file_path)

for model in GPT_Models:
    print(f"Checking compatibility for model: {model}")
    
    use_temp = True
    try:
        # Pre-flight check call
        client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "test"}],
            temperature=0.0
        )
        # If no exception occurs, temperature is supported
        print(f"--> SUCCESS: {model} ALLOWS temperature parameter.")
        
    except Exception as e:
        # Check if the error is due to an unsupported temperature parameter
        if "temperature" in str(e).lower() or "unsupported_parameter" in str(e).lower():
            print(f"--> FAILURE: {model} DOES NOT ALLOW temperature parameter. Shifting permanently to no-temperature mode.")
            use_temp = False
        else:
            print(f"--> Connection check warning for {model}: {e}. Defaulting to temperature mode.")

    print(f"Starting classification loop for model: {model} (Temperature Mode: {use_temp})")
    column_name = f"LLM_{model}_Zero_Shot"
    
    # Apply to dataframe using the determined temperature setting
    df_q1[column_name] = df_q1['Text'].progress_apply(
        lambda x: classify_misinfo_new_prompt(x, model, use_temperature=use_temp)
    )
    
    # Checkpoint: Save immediately after this model completes
    df_q1.to_csv(file_path, index=False)
    # print(f"Saved checkpoint for {model} to '{output_filename}'\n" + "-"*50)

In [ ]:
import re

# 1. Reuse your exact model list to iterate over columns
GPT_Models = [
    "gpt-5.5-2026-04-23",
    "gpt-5.4-2026-03-05",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.2-2025-12-11",
    "gpt-5.1-2025-11-13",
    "gpt-5-2025-08-07",
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4.1-2025-04-14",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4.1-nano-2025-04-14",
    "gpt-4o-2024-11-20",
    "gpt-4o-mini-2024-07-18"
]

def extract_clean_label(text):
    # Convert to string and strip whitespace/quotes
    text_str = str(text).strip().replace("'", "").replace('"', '')
    
    # Check if the string or line starts with Yes / Label: Yes
    if text_str.startswith("Yes") or text_str.startswith("Label: Yes"):
        return "Yes"
    # Check if the string or line starts with No / Label: No
    elif text_str.startswith("No") or text_str.startswith("Label: No"):
        return "No"
    
    # Fallback regex search just in case there's an unexpected leading character or newline format
    match = re.search(r'\b(Yes|No)\b', text_str, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
        
    return "Flag" # Flag any anomalies or API errors

# 2. Process each model column and generate the new label column
for model in GPT_Models:
    src_col = f"LLM_{model}_Zero_Shot"
    new_col = f"LLM_{model}_Zero_Shot_Label"
    
    if src_col in df_q1.columns:
        test_set_experiments[new_col] = test_set_experiments[src_col].apply(extract_clean_label)

# 3. Verify a slice of your new label columns
label_cols = [f"LLM_{m}_Zero_Shot_Label" for m in GPT_Models if f"LLM_{m}_Zero_Shot_Label" in test_set_experiments.columns]
print(f"Created {len(label_cols)} clean label columns.")
test_set_experiments[label_cols].head()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

# 1. Define your ground truth and model lists
ground_truths = ['Final_Label']
GPT_Models = [
    "gpt-5.5-2026-04-23",
    "gpt-5.4-2026-03-05",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.2-2025-12-11",
    "gpt-5.1-2025-11-13",
    "gpt-5-2025-08-07",
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4.1-2025-04-14",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4.1-nano-2025-04-14",
    "gpt-4o-2024-11-20",
    "gpt-4o-mini-2024-07-18"
]

metrics_results = []

# 2. Iterate through ground truths and models to compute metrics
for gt in ground_truths:
    for model in GPT_Models:
        predictive_col = f"LLM_{model}_Zero_Shot_Label"
        
        # Ensure the column exists in the DataFrame before computing
        if predictive_col in test_set_experiments.columns:
            # Standardize to string and strip whitespace
            y_true = test_set_experiments[gt].astype(str).str.strip()
            y_pred = test_set_experiments[predictive_col].astype(str).str.strip()
            
            # Calculate standard weighted metrics
            precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            
            # Compute confusion matrix elements to get TPR and TNR
            # Note: labels=['No', 'Yes'] maps 'No' to index 0 and 'Yes' to index 1
            labels = ['No', 'Yes']
            cm = confusion_matrix(y_true, y_pred, labels=labels)
            
            # Extract values from the 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            
            # Calculate True Positive Rate (Sensitivity) and True Negative Rate (Specificity)
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            
            metrics_results.append({
                'Ground Truth': gt,
                'Model': model,
                'Precision (Weighted)': precision,
                'Recall (Weighted)': recall,
                'F1 Score (Weighted)': f1,
                'TPR (Sensitivity)': tpr,
                'TNR (Specificity)': tnr
            })

# 3. Create the initial summary table
experiments_metrics_df = pd.DataFrame(metrics_results)

# 4. Sort each Ground Truth section descending by F1 Score
sorted_metrics_df = experiments_metrics_df.sort_values(
    by=['Ground Truth', 'F1 Score (Weighted)'], 
    ascending=[True, False]
).reset_index(drop=True)

# 5. Find the exact row index where the first ground truth section ends
split_idx = sorted_metrics_df[sorted_metrics_df['Ground Truth'] == sorted_metrics_df['Ground Truth'].iloc[0]].index[-1]

# 6. Inject the solid black horizontal divider row line without adding colors
styled_experiments_df = sorted_metrics_df.style.set_table_styles([
    {
        'selector': f'tbody tr:nth-child({split_idx + 1})',
        'props': [
            ('border-bottom', '3px solid #000000'), 
            ('box-shadow', '0 2px 0 #000000')
        ]
    }
], overwrite=False)

# Display the final styled table
styled_experiments_df

# Task 1: Few Shots

In [ ]:
few_shots_dict = {}

# Group by the clean binary label
grouped = sampled_df.groupby('Final_Label')

# Generate the shots for k = 1 to 5
for k in range(1, 6):
    # Sample k rows from each group and combine them
    k_shot_df = grouped.apply(lambda x: x.sample(n=k, random_state=42)).reset_index(drop=True)
    
    # Store the dataframe (or you can store specific text/label columns)
    few_shots_dict[k] = k_shot_df

# ---- Verification Display ----
for k, df in few_shots_dict.items():
    print(f"k = {k} | Total rows: {len(df)}")
    print(df['Final_Label'].value_counts())
    print("-" * 30)

In [ ]:
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# 1. Configuration and Lists
ground_truths = ['Final_Label']
GPT_Models = [
    "gpt-5.5-2026-04-23",
    "gpt-5.4-2026-03-05",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.2-2025-12-11",
    "gpt-5.1-2025-11-13",
    "gpt-5-2025-08-07",
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4.1-2025-04-14",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4.1-nano-2025-04-14",
    "gpt-4o-2024-11-20",
    "gpt-4o-mini-2024-07-18"
]

output_filename = "Camera_Ready_GT_few_shot_experiments_results.csv"
tqdm.pandas()

# 2. Main Classification Function with Dynamic Few-Shot Prompt Injection
def classify_misinfo_few_shot(text, model_name, prompt, use_temperature=True):
    retries = 2
    while retries > 0:
        try:
            kwargs = {
                "model": model_name,
                "messages": [{"role": "user", "content": prompt + text}]
            }
            if use_temperature:
                kwargs["temperature"] = 0.0
                
            completion = client.chat.completions.create(**kwargs)
            return completion.choices[0].message.content
            
        except Exception as e:
            print(f"Error with {model_name}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)
            
    return "API_ERROR"

# ==========================================
# OUTER LOOP: Iterate over k = 1 to 5 shots
# ==========================================
for k in range(1, 6):
    print(f"\n" + "="*60)
    print(f"   STARTING EXPERIMENTS FOR K = {k} SHOTS")
    print("="*60)
    
    # Extract the dataframe containing k-Yes and k-No examples
    k_shot_df = few_shots_dict[k]
    
    # Build the few-shot examples block to append to your original prompt instructions
    few_shot_examples_block = "\n\nHere are some examples of texts and their correct label for misinfo:\n"
    for idx, row in k_shot_df.iterrows():
        example_text = row['Text']
        example_label = row['Final_Label']
        
        few_shot_examples_block += f"\nExample {idx+1}:\n"
        few_shot_examples_block += f"Text: {example_text}\n"
        few_shot_examples_block += f"Label: {example_label}\n"
        few_shot_examples_block += "-" * 20 + "\n"
    
    # Construct the final unified prompt structure for this iteration
    few_shot_prompt_base = Task1_Misinformation_Prompt_merged + few_shot_examples_block + "\nNow, analyze the following text and provide the output using the exact same format rules:\nHere is the text:\n"
    
    # ------------------------------------------
    # INNER LOOP: Run the prompt across all 14 models
    # ------------------------------------------
    for model in GPT_Models:
        print(f"Checking compatibility for model: {model} at k={k}")
        
        # Pre-flight check call for temperature compatibility
        use_temp = True
        try:
            client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": "test"}],
                temperature=0.0
            )
            print(f"--> SUCCESS: {model} ALLOWS temperature parameter.")
        except Exception as e:
            if "temperature" in str(e).lower() or "unsupported_parameter" in str(e).lower():
                print(f"--> FAILURE: {model} DOES NOT ALLOW temperature parameter. Shifting permanently to no-temperature mode.")
                use_temp = False
            else:
                print(f"--> Connection check warning for {model}: {e}. Defaulting to temperature mode.")

        # Define column naming convention for this batch
        column_name = f"LLM_{model}_K_{k}_Shot"
        print(f"Starting classification loop for model: {model} (K={k}, Temp Mode: {use_temp})")
        
        # Apply to your dataset
        test_set_experiments[column_name] = test_set_experiments['Text'].progress_apply(
            lambda x: classify_misinfo_few_shot(x, model, few_shot_prompt_base, use_temperature=use_temp)
        )
        
        # Checkpoint: Save progress immediately after each model finishes processing its rows
        test_set_experiments.to_csv(output_filename, index=False)
        print(f"Saved checkpoint for {model} (k={k}) to '{output_filename}'\n" + "-"*50)

In [ ]:
import re


def extract_clean_label(text):
    text_str = str(text).strip().replace("'", "").replace('"', '')
    
    if text_str.startswith("Yes") or text_str.startswith("Label: Yes"):
        return "Yes"
    elif text_str.startswith("No") or text_str.startswith("Label: No"):
        return "No"
    
    match = re.search(r'\b(Yes|No)\b', text_str, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
        
    return "Flag"

# Loop through all 5 k-shot levels and all 14 models
for k in range(1, 6):
    for model in GPT_Models:
        src_col = f"LLM_{model}_K_{k}_Shot"
        new_col = f"LLM_{model}_K_{k}_Shot_Label"
        
        if src_col in test_set_experiments.columns:
            test_set_experiments[new_col] = test_set_experiments[src_col].apply(extract_clean_label)


# 3. Verify a slice of your new few-shot label columns
label_cols = [
    f"LLM_{m}_K_{k}_Shot_Label" 
    for k in range(1, 6) 
    for m in GPT_Models 
    if f"LLM_{m}_K_{k}_Shot_Label" in test_set_experiments.columns
]

print(f"Created {len(label_cols)} clean few-shot label columns.")
test_set_experiments[label_cols].head()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

# 1. Define your ground truth, k-shots, and model lists
ground_truths = ['Final_Label']
test_set_experiments.loc[test_set_experiments['Post ID'] == 11, 'Final_Label'] = 'No'

GPT_Models = [
    "gpt-5.5-2026-04-23",
    "gpt-5.4-2026-03-05",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.2-2025-12-11",
    "gpt-5.1-2025-11-13",
    "gpt-5-2025-08-07",
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4.1-2025-04-14",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4.1-nano-2025-04-14",
    "gpt-4o-2024-11-20",
    "gpt-4o-mini-2024-07-18"
]

metrics_results = []

# 2. Iterate through ground truths, k-shots, and models to compute metrics
for gt in ground_truths:
    for k in range(1, 6):
        for model in GPT_Models:
            predictive_col = f"LLM_{model}_K_{k}_Shot_Label"
            
            if predictive_col in test_set_experiments.columns:
                y_true = test_set_experiments[gt].astype(str).str.strip()
                y_pred = test_set_experiments[predictive_col].astype(str).str.strip()
                
                precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
                recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
                f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
                
                labels = ['No', 'Yes']
                cm = confusion_matrix(y_true, y_pred, labels=labels)
                tn, fp, fn, tp = cm.ravel()
                
                tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
                
                metrics_results.append({
                    'Ground Truth': gt,
                    'K_Shots': k,
                    'Model': model,
                    'Precision (Weighted)': precision,
                    'Recall (Weighted)': recall,
                    'F1 Score (Weighted)': f1,
                    'TPR (Sensitivity)': tpr,
                    'TNR (Specificity)': tnr
                })

# 3. Create the summary DataFrame
experiments_metrics_df = pd.DataFrame(metrics_results)

# 4. Sort strictly by Ground Truth first, then purely by F1 Score descending across all shots
sorted_metrics_df = experiments_metrics_df.sort_values(
    by=['Ground Truth', 'F1 Score (Weighted)'], 
    ascending=[True, False]
).reset_index(drop=True)

# 5. Track styling configurations
table_styles = []

# Find the single row index where q1_label_file1 ends and q1_label_file2 begins
gt_split_idx = sorted_metrics_df[sorted_metrics_df['Ground Truth'] == sorted_metrics_df['Ground Truth'].iloc[0]].index[-1]

# 6. Loop through rows to identify boundaries where K_Shots changes or GT changes
for idx in range(len(sorted_metrics_df) - 1):
    # Thick black line where Ground Truth changes
    if idx == gt_split_idx:
        table_styles.append({
            'selector': f'tbody tr:nth-child({idx + 1})',
            'props': [
                ('border-bottom', '6px solid #000000'), 
                ('box-shadow', '0 4px 0 #000000')
            ]
        })
    # Green line whenever the K_Shots value differs between adjacent sorted rows
    elif sorted_metrics_df.loc[idx, 'K_Shots'] != sorted_metrics_df.loc[idx + 1, 'K_Shots']:
        table_styles.append({
            'selector': f'tbody tr:nth-child({idx + 1})',
            'props': [
                ('border-bottom', '2.5px solid #2ca02c'), 
                ('box-shadow', '0 1.5px 0 #2ca02c')
            ]
        })

# 7. Apply the compiled line styles to the table
styled_experiments_df = sorted_metrics_df.style.set_table_styles(table_styles, overwrite=False)

# Display the final cross-shot sorted metrics table
styled_experiments_df

# Other Tasks

In [ ]:
prompts_dict = {

    "task1": """
You are a medical expert, specialized in identifying cancer-related misinformation. 
Definition:
Cancer misinformation refers to any information about cancer, its risk factors, prevention, diagnosis, or treatment that is false, inaccurate, misleading, or not supported by current scientific evidence or consensus, and that may lead to harmful consequences. This includes:
Unproven or ineffective treatments
Misrepresentation or selective use of facts
Outdated or decontextualized medical information
Conspiracy narratives (e.g., hidden cures)
Promotional or commercial content presented as medical advice

Task: Analyze the text provided below. Classify it as either:
'Yes' if the text contains misinformation
'No' if the text does not contain misinformation or if there is insufficient information


Output Format:
Label: 'Yes' or 'No' . Don't use additional text, and don't use [] around the label
Evidence: If classified as 1, quote the specific part of the text that suggests or implies misinformation and explain why it contradicts scientific evidence.
If classified as 1, explain your step-by-step reasoning to conclude that the text contains misinformation
""",

    "task2": """
You are an expert medical data annotator.

Your task is to classify the text into the most relevant stage of the cancer care journey.

Categories:

- Prevention/Screening:
Risk factors, lifestyle, prevention behaviors, early detection, screening, mammograms, colonoscopies, genetic risk, or preventive advice.

- Treatment:
Medical interventions such as chemotherapy, radiation, surgery, immunotherapy, medications, or active treatment decisions.

- Other/Unclear/NA:
Diagnosis, survivorship, post-treatment experiences, emotional support, unclear references, unrelated discussion, or insufficient information.

Instructions:
- Choose the single BEST category.
- Focus on the primary topic of the text.
- If multiple stages are mentioned, prioritize the main emphasis.

Output format:
Prevention/Screening
OR
Treatment
OR
Other/Unclear/NA

No additional words.

Here is the text:
""",

    "task3": """
You are an expert medical data annotator.

Your task is to classify the communication style of the text regarding cancer misinformation.

Definitions:

- Seeking:
The user is asking for advice, clarification, information, recommendations, or a second opinion.

- Sharing:
The user is providing information, opinions, claims, experiences, or statements.

- Both:
The text simultaneously includes both seeking and sharing behavior.

Instructions:
- Focus on the author's communicative intent.
- Questions alone are not always "Seeking" if the text mainly shares information.
- If the text strongly contains both behaviors, choose "Both".

Output format:
Seeking
OR
Sharing
OR
Both

No additional words.

Here is the text:
""",

    "task4": """
You are an expert medical data annotator.

Your task is to classify the PRIMARY type of cancer-related misinformation present in the text.

Definitions:

- Fabricated Content:
Completely false information with no factual basis.

- Misleading Content:
Information that distorts, exaggerates, oversimplifies, or selectively presents facts.

- False Context:
True information shared in an incorrect, misleading, or inappropriate context.

- False Connection:
The headline, claim, or conclusion does not match the supporting content.

- Imposter Content:
Fake, impersonated, or falsely attributed credible sources.

- Manipulated Content:
Altered images, videos, audio, statistics, or data.

- Satire/Parody:
Content intended humorously or satirically that could be misunderstood as factual.

- Other:
Does not clearly fit the categories above.

Instructions:
- Select the dominant misinformation type.
- Focus on HOW the misinformation is constructed, not the topic itself.
- If no clear misinformation type is identifiable, select "Other".

Output format:
[Label]

No additional words.

Here is the text:
""",

    "task5": """
You are an expert medical data annotator.

Your task is to classify the potential health risk level of the misinformation in the text.

Definitions:

1 - Low Risk:
Content with little or no likely impact on health behavior or outcomes; trivial, vague, or unlikely to influence medical decisions.

2 - Moderate Risk:
Content that may influence beliefs, perceptions, or minor health-related decisions.

3 - High Risk:
Content likely to encourage harmful health decisions, treatment avoidance, delayed care, dangerous alternative treatments, or potentially life-threatening behavior.

Instructions:
- Evaluate the likely real-world consequences if someone believed or acted on the misinformation.
- Consider severity, immediacy, and potential harm.
- Focus on potential behavioral impact, not offensiveness.

Output format:
Low Risk
OR
Moderate Risk
OR
High Risk

No additional words.

Here is the text:
""",

    "task6": """
You are an expert medical data annotator.

Your task is to classify the stance expressed toward cancer-related misinformation.

Definitions:

1) Accepts:
The text explicitly endorses, believes, supports, or promotes the misinformation claim.

2) Accept + Question:
The text leans toward accepting or considering the misinformation while expressing minor hesitation or asking clarifying questions.

3) Rejects:
The text explicitly refutes, debunks, criticizes, or disagrees with the misinformation claim.

4) Questions:
The text expresses curiosity, uncertainty, skepticism, or asks questions without clearly accepting or rejecting the claim.

5) Reject + Question:
The text fundamentally rejects the misinformation but also asks a related follow-up question.

6) Other:
Does not clearly fit the categories above.


Instructions:
- Focus on the author's stance toward the misinformation itself.
- Distinguish genuine curiosity from endorsement.
- Choose the closest overall stance.

Output format:
Accepts
OR
Accept + Question
OR
Rejects
OR
Questions
OR
Reject + Question
OR 
Other

No additional words.

Here is the text:
""",

    "task7": """
You are an expert medical data annotator.

Your task is to classify the PRIMARY topic of cancer-related misinformation discussed in the text.

Categories:

1) Causation, risk factors & prevention claims
(Unproven causes of cancer, misrepresented risk factors, unsupported prevention methods, e.g., "sugar causes cancer," "alkaline diet prevents cancer," "stress causes tumors.")

2) Diagnosis & screening claims
(Claims about symptoms, self-diagnosis, screening tests being dangerous, inaccurate, unnecessary, or misleading.)

3) Safety & effectiveness of conventional treatments
(Claims that chemotherapy, radiation, surgery, or medical treatments are ineffective, harmful, toxic, unnecessary, or intentionally deceptive.)

4) Unproven or alternative treatments
(Herbs, supplements, diets, naturopathy, homeopathy, detoxes, "natural cures," or non-evidence-based alternatives presented as cancer treatments.)

5) Conspiracy & institutional distrust
(Big Pharma conspiracies, hidden cures, government cover-ups, suppressed research, distrust of doctors or institutions.)

6) Morality, religion & ideology
(Religious, spiritual, karmic, moral, or ideological explanations or objections related to cancer and treatment.)

7) Medical autonomy & medical freedom
(Rights-based arguments about refusing treatment, parental authority, bodily autonomy, or distrust of medical mandates.)

8) Other:
Does not clearly fit the categories above.

Instructions:
- Select the SINGLE dominant topic.
- Focus on the central misinformation theme.
- If multiple topics appear, choose the primary emphasis.

Output format:
[Label]

No additional words.

Here is the text:
"""
}

In [ ]:
import pandas as pd

fewshot_prompts = {}
tasks = ["Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]

for task in tasks:
    task_num = int(task.replace("Q", ""))
    label_col = f"Final_Label_{task}"
    dict_key = f"task{task_num}"
    base_prompt = prompts_dict[dict_key]

    # 1. Pre-sample up to 4 shots per label once for this task
    master_shots = {}
    labels = misinfo_data[label_col].dropna().unique()

    for label in labels:
        label_df = misinfo_data[misinfo_data[label_col] == label]
        # Draw up to 4 total examples for this label (or max available)
        max_draw = min(4, len(label_df))
        if max_draw > 0:
            master_shots[label] = label_df.sample(n=max_draw, random_state=42)
        else:
            master_shots[label] = pd.DataFrame()

    # 2. Build prompts for k = 1 to 4 using cumulative slices of the master pool
    for k in range(1, 5):
        shot_text_blocks = []

        for label, label_pool in master_shots.items():
            if label_pool.empty:
                continue

            # Take the first 'k' rows from the master pool for this label
            k_shots = label_pool.head(k)

            for _, row in k_shots.iterrows():
                # Adjust 'text' if your column is named 'Text' or 'post'
                shot_text_blocks.append(f"Text: {row['Text']}\nLabel: {label}")

        # Combine shot blocks
        shot_examples_text = "\n---\n".join(shot_text_blocks)

        # Construct final prompt
        few_shot_prompt = (
            f"{base_prompt}\n\n"
            f"Here are some examples to guide your classification:\n"
            f"===\n{shot_examples_text}\n"
            f"Now, classify the following text:\n"
        )

        # Store in 2D dictionary: fewshot_prompts[(task_num, k)]
        fewshot_prompts[(task_num, k)] = few_shot_prompt

In [ ]:
# Retreieve Remaining Posts in Each Task

# Assume 'master_shots' or the sampled indices from the loop were collected.
# Here is the complete standalone script to build 'remaining_data_dict':

remaining_data_dict = {}
tasks = ["Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]

for task in tasks:
    task_num = int(task.replace("Q", ""))
    label_col = f"Final_Label_{task}"

    # Collect all indices used as few-shot examples across ALL labels for this task
    used_indices = set()

    labels = misinfo_data[label_col].dropna().unique()
    for label in labels:
        label_df = misinfo_data[misinfo_data[label_col] == label]

        # Get the up-to-4 examples sampled for this label
        max_draw = min(4, len(label_df))
        if max_draw > 0:
            label_shots = label_df.sample(n=max_draw, random_state=42)
            used_indices.update(label_shots.index)

    # Filter misinfo_data to keep only rows NOT used in any shot for this task
    remaining_df = misinfo_data.drop(index=list(used_indices))

    # Store in dictionary by task number (e.g., key=2 for Q2)
    remaining_data_dict[task_num] = remaining_df

    print(
        f"Task Q{task_num}: Original = {len(misinfo_data)} | "
        f"Shots Used = {len(used_indices)} | "
        f"Remaining = {len(remaining_df)}"
    )

In [ ]:
import pandas as pd

remaining_files_dict = {}

# Save each task's remaining DataFrame to a CSV file
for task_num, df_remaining in remaining_data_dict.items():
    file_path = f"remaining_misinfo_data_q{task_num}.csv"

    # Save to disk
    df_remaining.to_csv(file_path, index=True)

    # Store file path in dictionary
    remaining_files_dict[task_num] = file_path

print("Saved files:")
for task_num, path in remaining_files_dict.items():
    print(f"Task Q{task_num}: {path}")

In [ ]:
# Zero Shot Prompt

import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

tqdm.pandas()


# =====================================================
# SETTINGS
# =====================================================

model_selected = "gpt-4o-2024-11-20"
text_col = "Text"  # Adjust if your text column has a different name

# Tasks to run: task 2 through 7
task_numbers = range(2, 8)

# Dictionary to store updated DataFrames in memory
task_results = {}


# =====================================================
# CLASSIFICATION FUNCTION
# =====================================================


def classify_tasks_prompt(text, prompt, retries=2):
    while retries > 0:
        try:
            completion = client.chat.completions.create(
                model=model_selected, temperature=0,
                messages=[{"role": "user", "content": prompt + str(text)}],
            )
            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_selected}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"


# =====================================================
# RUN TASKS 2–7 ON REMAINING DATA
# =====================================================

for k in task_numbers:
    print("\n" + "=" * 70)
    print(f"STARTING TASK {k}")
    print("=" * 70)

    # 1. Get base prompt for current task
    prompt_key = f"task{k}"
    base_prompt = prompts_dict[prompt_key]

    # 2. Extract remaining DataFrame for current task
    data_task = remaining_data_dict[k].copy()
    print(
        f"Rows in remaining_data_dict for task {k}: {len(data_task)}"
    )

    # 3. Define LLM prediction column name
    prediction_col = f"LLM_prediction_task_{k}"

    # 4. Generate LLM predictions on remaining dataset
    data_task[prediction_col] = data_task[text_col].progress_apply(
        lambda x: classify_tasks_prompt(x, base_prompt)
    )

    # 5. Store DataFrame in dictionary
    task_results[k] = data_task

    # 6. Overwrite the task's CSV file with prediction column added
    output_file = f"remaining_misinfo_data_q{k}.csv"
    data_task.to_csv(output_file, index=True)

    print(f"Saved zero-shot task {k} results to: {output_file}")

print("\nAll tasks completed successfully.")

In [ ]:
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

tqdm.pandas()

# =====================================================
# SETTINGS
# =====================================================

model_selected = "gpt-4o-2024-11-20"
text_col = "Text"  # Adjust if your text column has a different name

task_numbers = range(2, 8)  # Tasks 2 to 7
k_values = range(1, 5)  # k = 1, 2, 3, 4


# =====================================================
# CLASSIFICATION FUNCTION
# =====================================================


def classify_tasks_prompt(text, prompt, retries=2):
    while retries > 0:
        try:
            completion = client.chat.completions.create(
                model=model_selected,
                temperature=0,  # Strict determinism
                messages=[{"role": "user", "content": prompt + str(text)}],
            )
            return completion.choices[0].message.content

        except Exception as e:
            print(f"Error with {model_selected}: {e}. Retrying...")
            retries -= 1
            time.sleep(5)

    return "API_ERROR"


# =====================================================
# NESTED LOOP: TASKS 2–7 x SHOTS k=1..4
# =====================================================

for task_num in task_numbers:
    # 1. Load current task's CSV (contains zero-shot predictions)
    file_path = f"remaining_misinfo_data_q{task_num}.csv"
    data_task = pd.read_csv(file_path, index_col=0)

    # Track length of remaining data
    task_len = len(data_task)

    print("\n" + "=" * 70)
    print(f"STARTING FEW-SHOT RUNS FOR TASK {task_num} (N = {task_len} rows)")
    print("=" * 70)

    # 2. Iterate through k = 1 to 4
    for k in k_values:
        print(f"\n--- Task Q{task_num} | k = {k} | Evaluated on {task_len} rows ---")

        # Get the corresponding embedded few-shot prompt
        few_shot_prompt = fewshot_prompts[(task_num, k)]

        # Define column name for this shot setting
        pred_col = f"LLM_prediction_task_{task_num}_k{k}"

        # Run predictions
        data_task[pred_col] = data_task[text_col].progress_apply(
            lambda x: classify_tasks_prompt(x, few_shot_prompt)
        )

    # 3. Overwrite file with remaining data length, zero-shot, and few-shot predictions
    data_task.to_csv(file_path, index=True)
    print(f"\nSuccessfully updated {file_path} (Total Rows: {task_len})")

print("\nAll few-shot runs completed successfully.")

In [ ]:
import pandas as pd

task_numbers = range(2, 8)  # Tasks 2 to 7
k_values = range(1, 5)  # k = 1 to 4


def clean_label(val):
    if pd.isna(val):
        return val
    # Convert to string and strip brackets, quotes, whitespace, and trailing periods
    return str(val).strip("[]'\" .")


for task_num in task_numbers:
    file_path = f"remaining_misinfo_data_q{task_num}.csv"

    # Load dataset for current task
    df_task = pd.read_csv(file_path, index_col=0)

    print("\n" + "=" * 70)
    print(
        f"CLEANED LABEL DISTRIBUTIONS FOR TASK Q{task_num} (Total Rows: {len(df_task)})"
    )
    print("=" * 70)

    # 1. Zero-shot distribution
    zero_shot_col = f"LLM_prediction_task_{task_num}"
    if zero_shot_col in df_task.columns:
        cleaned_series = df_task[zero_shot_col].apply(clean_label)
        print(f"\n[Zero-Shot] Column: {zero_shot_col}")
        print(cleaned_series.value_counts(dropna=False))

    # 2. Few-shot distributions (k = 1 to 4)
    for k in k_values:
        few_shot_col = f"LLM_prediction_task_{task_num}_k{k}"
        if few_shot_col in df_task.columns:
            cleaned_series = df_task[few_shot_col].apply(clean_label)
            print(f"\n[Few-Shot k={k}] Column: {few_shot_col}")
            print(cleaned_series.value_counts(dropna=False))